# Lab 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import scipy.optimize as optim

In [ ]:
data = np.loadtxt('TwoSlitData.csv', delimiter=',', skiprows=1)
headers = ["x", "uncertainty_in_x", "Intensity"]

x = data[:, 0]
uncertainty = data[:, 1]
intensity = data[:, 2]
pi = np.pi

In [ ]:
peaks, _ = signal.find_peaks(intensity, distance=3, height=0.5)
c3c4 = 50. # Best guess from binary search, starting value 2000

plt.plot(x, intensity, label="intensity")
plt.scatter(x[peaks], intensity[peaks], label='peaks')
plt.plot(x, np.sinc(pi * c3c4 * x))

In [ ]:
# Initial guess: use the peak locations found by scipy.signal
def f(x, c1, c2, c3):
    return c1 * np.cos(pi*c2*x)**2 * np.sinc(pi*c3*x)**2

c1_start, c2_start, c3_start = 1., 2000., 50.
popt, pcov = optim.curve_fit(f, 
                             xdata=x, 
                             ydata=intensity, 
                             p0=[c1_start, c2_start, c3_start])
c1, c2, c3 = popt

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x, intensity, marker='o', s=5, label="raw intensity")
ax.plot(x, f(x, c1, c2, c3), label="initial fit")
ax.plot(x, f(x, c1, c2, c3), label="optimized fit")
ax.legend()
ax.set_title("Double-slit intensity data")
ax.set_xlabel("Horizontal displacement (x), unknown units")
ax.set_ylabel("Relative intensity")
fig.show()

In [ ]:
## Residual plot
fig, ax = plt.subplots()
ax.scatter(x, intensity - f(x, c1, c2, c3))
ax.set_title("Residual plot")
ax.set_xlabel("Horizontal displacement (x), unknown units")
ax.set_ylabel("Residual of Intensity (optimized fit - data)")

In [ ]:
print(f"Final parameter values are {c1=}, {c2=}, {c3=}")

In [ ]:
mse_start = np.mean(np.square(intensity - f(x, c1_start, c2_start, c3_start)))
mse = np.mean(np.square(intensity - f(x, c1, c2, c3)))
print(f"The MSE of the initial guess fit was {mse_start}")
print(f"The MSE of the optimized fit was {mse}")

Comment: The fit looks accurate in the middle, but the fit envelope rolls off faster than the raw data at the peak of the 4th left and right bright fringes. 

Explanations I can come up with:
1. One outlier at the center? Unlikely, the fit is clearly off at 5 peak points on the 4th bright fringes, and only one at the center. Least-squares would not overweight the central outlier.
2. Small angle approximation is not accurate?
3. Large ratio d/L (slot spacing over distance to screen) changes the rolloff properties?

In [ ]:
# Residual plot
